# Colorimetry Analysis Workflow

This notebook implements a complete colorimetry pipeline based on the **CIE 1931 2-Degree Standard Observer** and the **D65 illuminant**. It processes raw reflectance spectral data, computes tristimulus XYZ values, converts them to CIELab (L\*, a\*, b\*) and sRGB color spaces, and produces publication-quality visualizations.

## Section 1 – Import Libraries

All external packages required by this notebook are imported here. Run this cell first before executing any other cell.

In [ ]:
import colour
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os
import io

## Section 2 – Sample Separation

The raw CSV files exported by most spectrophotometers contain one column-pair per sample (wavelength | %R | wavelength | %R | …). This cell:

1. Loads the raw CSV specified by `INPUT_FILE`.
2. Extracts the first wavelength column and all reflectance columns (every other column starting at index 1).
3. Assembles a tidy DataFrame (`master_data`) with one wavelength column and one reflectance column per sample, removing any non-numeric rows (e.g. summary text appended by the instrument).
4. (Optional) Drops the first column if it's a 100%T baseline reference rather than a real sample.
5. Saves the result to `data-processed/` with a `_processed` suffix.

> **What to change:**
> - `INPUT_FILE` to the path of your raw CSV file (relative or absolute).
> - `DROP_BASELINE` to `True` if your instrument records a 100%T baseline as the first column.

In [ ]:
# ── USER PARAMETER ────────────────────────────────────────────────────────────
INPUT_FILE = 'data-raw/image1_data_cardstock_prueba.csv'   # <-- change this path
DROP_BASELINE = True  # <-- set to True if your spectrophotometer prepends a 100%T baseline column
# ─────────────────────────────────────────────────────────────────────────────

# Load raw data
data = pd.read_csv(INPUT_FILE, low_memory=False)

# Extract wavelength column (first column, skip header row at index 0)
wavelengths = data.iloc[1:, 0]

# Extract reflectance columns (every other column starting from index 1)
reflectances = data.iloc[1:, 1::2]

# Assemble tidy DataFrame
master_data = pd.concat([wavelengths, reflectances], axis=1)

if DROP_BASELINE:
    master_data = master_data.drop(master_data.columns[1], axis=1)

num_samples = master_data.shape[1] - 1

# Rename columns
master_data.columns = ['Wavelength (nm)'] + [
    f'Reflectance POS {i} (%R)' for i in range(num_samples)
]

# Drop non-numeric rows (instrument summary text becomes NaN after coercion)
master_data['Wavelength (nm)'] = pd.to_numeric(
    master_data['Wavelength (nm)'], errors='coerce'
)
master_data = master_data.dropna(subset=['Wavelength (nm)'])

# Save processed file
os.makedirs('data-processed', exist_ok=True)
base_name     = os.path.splitext(os.path.basename(INPUT_FILE))[0]
OUTPUT_FILE   = f'data-processed/{base_name}_processed.csv'
master_data.to_csv(OUTPUT_FILE, index=False)

print(f'Processed file saved: {OUTPUT_FILE}')
print(f'  Samples found : {num_samples}')
print(f'  Wavelength rows: {len(master_data)}')

## Section 3 – View Data

Quick inspection of the raw file structure and the processed DataFrame to confirm the reshaping worked correctly.

- **Raw data** (`data.head()`) shows the original interleaved column layout.
- **Processed data** (`master_data.head()`) shows the clean, tidy layout ready for colorimetry.

In [ ]:
print('=== Raw data (original layout) ===')
display(data.head())

print('\n=== Processed data (tidy layout) ===')
display(master_data.head())

## Section 4 – Load Processed Data

This cell loads the processed CSV that will be used for colorimetry calculations. You can either:

- Point `FILE_PATH` to a file generated by Section 2, or
- Point it directly to any pre-processed CSV that already has the tidy format (one wavelength column + one reflectance column per sample).

A clean `file_name` string (without path or `_processed` suffix) is also derived here for use as a prefix in output file names.

> **What to change:** Set `FILE_PATH` to the processed CSV you want to analyse.

In [ ]:
# ── USER PARAMETER ────────────────────────────────────────────────────────────
FILE_PATH = 'data-processed/image1_data_cardstock_prueba_processed.csv'  # <-- change
# ─────────────────────────────────────────────────────────────────────────────

data_processed = pd.read_csv(FILE_PATH)

# Derive a clean file-name prefix for output files
file_name = os.path.splitext(os.path.basename(FILE_PATH))[0].replace('_processed', '')

print(f'Loaded: {FILE_PATH}')
print(f'Samples : {data_processed.shape[1] - 1}')
print(f'Output prefix: {file_name}')

## Section 5 – Reflectance Spectrum Visualization

Plots the spectral reflectance curves (%R vs wavelength) for all samples in the processed file.

Each sample is drawn as a semi-transparent line so overlapping curves remain readable. The figure is exported as an SVG to `results/graphics/reflectance_spectrum/`.

> **What to change:**
> - `WL_MIN` / `WL_MAX` – wavelength range to display (nm).
> - Line colour, alpha, and figure size if you need to customize the appearance.

In [ ]:
# ── USER PARAMETERS ───────────────────────────────────────────────────────────
WL_MIN = 360   # minimum wavelength to display (nm)
WL_MAX = 750   # maximum wavelength to display (nm)
# ─────────────────────────────────────────────────────────────────────────────

wl_col     = 'Wavelength (nm)'
ref_cols   = [c for c in data_processed.columns if c != wl_col]
wavelength = data_processed[wl_col].values

# Filter to the desired wavelength range
mask = (wavelength >= WL_MIN) & (wavelength <= WL_MAX)

fig, ax = plt.subplots(figsize=(10, 6))
for col in ref_cols:
    ax.plot(wavelength[mask], data_processed[col].values[mask])

ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance (%R)')
ax.set_title(f'Reflectance Spectrum Raw (%R)')
ax.set_xlim(WL_MIN, WL_MAX)
plt.tight_layout()

# Save figure
out_dir = 'results/graphics/reflectance_spectrum'
os.makedirs(out_dir, exist_ok=True)
svg_path = f'{out_dir}/reflectance_spectrum_{file_name}.svg'
plt.savefig(svg_path, format='svg', bbox_inches='tight')
print(f'Figure saved: {svg_path}')
plt.show()

## Section 6 – Normalization & MultiSpectralDistribution

Before feeding data into the `colour` library two steps are required:

1. **Normalization** – the reflectance values are in percent (%R, 0–100). The `colour` library expects values in the [0, 1] range, so all columns are divided by 100.
2. **MultiSpectralDistribution (MSDS)** – the normalized DataFrame is wrapped in a `colour.MultiSpectralDistributions` object and aligned (interpolated + extrapolated) to a uniform 1 nm grid covering the visible range (360–750 nm by default). This alignment is mandatory because the CMF and illuminant data used later are defined on this standard grid.

> **What to change:**
> - `SHAPE_START` / `SHAPE_STOP` / `SHAPE_STEP` – spectral shape for alignment. Keep 360–750 nm at 1 nm unless you have a specific reason to change it.

In [ ]:
# ── USER PARAMETERS ───────────────────────────────────────────────────────────
SHAPE_START = 360   # start wavelength for spectral alignment (nm)
SHAPE_STOP  = 750   # stop  wavelength for spectral alignment (nm)
SHAPE_STEP  =   1   # wavelength interval (nm)
# ─────────────────────────────────────────────────────────────────────────────

# Set wavelength as DataFrame index for colour compatibility
data_indexed = data_processed.copy(deep=True)
data_indexed = data_indexed.set_index('Wavelength (nm)')
data_indexed = data_indexed.sort_index(ascending=True)

# Convert %R → [0, 1]
data_normalized = data_indexed / 100

# Build MultiSpectralDistributions object
msds = colour.MultiSpectralDistributions(data_normalized)

print(f'Original spectral shape : {msds.shape}')
print(f'First 5 wavelengths (nm): {msds.domain[:5]}')

# Align to the target spectral shape (interpolates to 1 nm steps)
shape = colour.SpectralShape(SHAPE_START, SHAPE_STOP, SHAPE_STEP)
msds  = msds.align(shape)

print(f'Aligned spectral shape  : {msds.shape}')

## Section 7 – Calculate XYZ Tristimulus Values

The XYZ tristimulus values are computed by integrating each reflectance spectrum against the CIE 1931 2-Degree Standard Observer colour-matching functions (CMFs) under the D65 illuminant.

The integration is performed by `colour.msds_to_XYZ` using the standard formula:

$$X = k \int S(\lambda)\, R(\lambda)\, \bar{x}(\lambda)\, d\lambda, \quad
   Y = k \int S(\lambda)\, R(\lambda)\, \bar{y}(\lambda)\, d\lambda, \quad
   Z = k \int S(\lambda)\, R(\lambda)\, \bar{z}(\lambda)\, d\lambda$$

where $S(\lambda)$ is the illuminant power, $R(\lambda)$ the reflectance, and $k$ is a normalisation constant.

The result is divided by 100 to express XYZ in the [0, 1] scale (where Y = 1 corresponds to a perfect white diffuser under D65).

> **Key variables:**
> - `cmfs` – CIE 1931 2-Degree CMFs (fixed for this workflow).
> - `illuminant` – D65 spectral power distribution (fixed for this workflow).
> - `XYZ` – array of shape `(n_samples, 3)` with (X, Y, Z) per sample.

In [ ]:
# CIE 1931 2-Degree Standard Observer colour-matching functions
cmfs = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']

# D65 illuminant spectral power distribution
illuminant = colour.SDS_ILLUMINANTS['D65']

# Compute XYZ tristimulus values (result divided by 100 → [0, 1] scale)
XYZ = colour.msds_to_XYZ(msds, cmfs, illuminant) / 100

print(f'XYZ array shape : {XYZ.shape}  →  (n_samples, 3)')
print(f'XYZ range       :  X [{XYZ[:,0].min():.4f}, {XYZ[:,0].max():.4f}]')
print(f'                   Y [{XYZ[:,1].min():.4f}, {XYZ[:,1].max():.4f}]')
print(f'                   Z [{XYZ[:,2].min():.4f}, {XYZ[:,2].max():.4f}]')

## Section 8 – Calculate CIELab (L\*, a\*, b\*)

CIELab is a perceptually uniform colour space where:
- **L\*** (0–100) – lightness (0 = black, 100 = white).
- **a\*** – green (negative) ↔ red (positive) axis.
- **b\*** – blue (negative) ↔ yellow (positive) axis.

The conversion from XYZ to CIELab requires the D65 white-point chromaticity (`illuminant_xy`) as reference. This is the standard white point for the CIE 1931 2-degree observer.

> **Key variable:** `Lab` – array of shape `(n_samples, 3)` with (L\*, a\*, b\*) per sample.

In [ ]:
# D65 white-point chromaticity (xy) for the CIE 1931 2-Degree Observer
illuminant_xy = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65']

# Convert XYZ → CIELab
Lab = colour.XYZ_to_Lab(XYZ, illuminant=illuminant_xy)

print('CIELab values computed.')
print(f'  L* range: [{Lab[:,0].min():.2f}, {Lab[:,0].max():.2f}]')
print(f'  a* range: [{Lab[:,1].min():.2f}, {Lab[:,1].max():.2f}]')
print(f'  b* range: [{Lab[:,2].min():.2f}, {Lab[:,2].max():.2f}]')

## Section 9 – Calculate sRGB

sRGB is the standard colour space for displaying colours on screens. The conversion from XYZ applies a chromatic adaptation (Bradford) and a non-linear gamma curve.

Because the sRGB gamut is smaller than the full CIE XYZ space, some samples may produce channel values outside [0, 1]. A scale gamut mapping is applied:
- Negative values are clipped to 0 (avoids imaginary colours).
- If the maximum channel value exceeds 1, all three channels are scaled down proportionally, which preserves hue while reducing saturation.

> **Key variables:**
> - `RGB` – raw sRGB values (may be out-of-gamut).
> - `RGB_mapped` – gamut-mapped sRGB values in [0, 1] (safe for display).

In [ ]:
# Convert XYZ → sRGB  (uses the same D65 white point)
RGB = colour.XYZ_to_sRGB(XYZ, illuminant=illuminant_xy)

# ── Gamut mapping (scale method – preserves hue, reduces saturation) ──────────
RGB_mapped       = np.clip(RGB, 0, None)                          # clip negatives
max_per_sample   = RGB_mapped.max(axis=1, keepdims=True)
scale            = np.where(max_per_sample > 1, max_per_sample, 1.0)
RGB_mapped       = RGB_mapped / scale                             # scale into [0,1]

n_out_of_gamut = (RGB.max(axis=1) > 1).sum()
print(f'sRGB values computed.')
print(f'  Samples out of sRGB gamut : {n_out_of_gamut} / {RGB.shape[0]}')

## Section 10 – Data Storage

Exports all computed colour-space values to Excel workbooks inside the `results/spaces_values/` directory:

| File | Contents |
|---|---|
| `XYZ/{file_name}_XYZ.xlsx` | X, Y, Z per sample |
| `LAB/{file_name}_LAB.xlsx` | L\*, a\*, b\* per sample |
| `MSDS/{file_name}_MSDS.xlsx` | Aligned reflectance spectra (wavelength × sample) |
| `sRGB/{file_name}_sRGB.xlsx` | R, G, B (raw, pre-gamut-mapping) per sample |

The row index is the sample label (column name from the processed CSV).

In [ ]:
# Build DataFrames for export
sample_labels = data_normalized.columns.tolist()

df_XYZ  = pd.DataFrame(XYZ,  columns=['X', 'Y', 'Z'],        index=sample_labels)
df_Lab  = pd.DataFrame(Lab,  columns=['L*', 'a*', 'b*'],      index=sample_labels)
df_sRGB = pd.DataFrame(RGB,  columns=['R', 'G', 'B'],         index=sample_labels)
df_msds = pd.DataFrame(msds.values, index=msds.domain,        columns=msds.labels)

# Create output directories and save
for sub in ['XYZ', 'LAB', 'MSDS', 'sRGB']:
    os.makedirs(f'results/spaces_values/{sub}', exist_ok=True)

df_XYZ.to_excel( f'results/spaces_values/XYZ/{file_name}_2XYZ.xlsx')
df_Lab.to_excel( f'results/spaces_values/LAB/{file_name}_2LAB.xlsx')
df_msds.to_excel(f'results/spaces_values/MSDS/{file_name}_2MSDS.xlsx')
df_sRGB.to_excel(f'results/spaces_values/sRGB/{file_name}_2sRGB.xlsx')

print('Excel files saved:')
for sub, suffix in [('XYZ','_XYZ'),('LAB','_LAB'),('MSDS','_MSDS'),('sRGB','_sRGB')]:
    print(f'  results/spaces_values/{sub}/{file_name}{suffix}.xlsx')

## Section 11 – CIE 1931 xy Chromaticity Diagram

Plots the CIE 1931 xy chromaticity diagram (horseshoe-shaped gamut boundary) with all sample chromaticities overlaid.

The (x, y) chromaticity coordinates are derived from XYZ as:

$$x = \frac{X}{X+Y+Z}, \quad y = \frac{Y}{X+Y+Z}$$

Samples close to the D65 white point (≈ 0.3127, 0.3290) are achromatic (grey/white). Samples near the spectral locus boundary are highly saturated.

The figure is saved as an SVG to `results/graphics/XYZ/`.

In [ ]:
# Compute xy chromaticity from XYZ
XYZ_sum = XYZ.sum(axis=1, keepdims=True)
xy      = XYZ[:, :2] / XYZ_sum   # x = X/(X+Y+Z),  y = Y/(X+Y+Z)

# Plot using colour's built-in diagram (returns axes for further customisation)
fig, ax = plt.subplots(figsize=(6, 7))
colour.plotting.plot_chromaticity_diagram_CIE1931(axes=ax, show=False)

ax.scatter(xy[:, 0], xy[:, 1],
           c=RGB_mapped, s=38, edgecolors='white', linewidths=0.5, label='Samples')

ax.set_title(f'CIE 1931 Chromaticity Diagram')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()

out_dir  = 'results/graphics/XYZ'
os.makedirs(out_dir, exist_ok=True)
svg_path = f'{out_dir}/cie1931_xy_{file_name}.svg'
plt.savefig(svg_path, format='svg', bbox_inches='tight')
print(f'Diagram saved: {svg_path}')
plt.show()

## Section 12 – CIELAB Channel Decomposition

Visualises how samples distribute across the three CIELAB channels:

1. **Step 1** – The gamut-mapped sRGB values are rendered as a colour grid (one square per sample).
2. **Step 2** – The rendered grid image is captured into memory, then converted back through sRGB → XYZ → CIELab. This lets us display the spatial distribution of each channel as a heatmap, making it easy to spot colour-shift patterns across the measurement grid.

**Colormaps:**
- L\* – greyscale (0 = black, 100 = white)
- a\* – green ↔ red (symmetric around 0)
- b\* – blue ↔ yellow (symmetric around 0)

> **What to change:** `COLS` controls how many samples appear per row in the colour grid.

In [ ]:
# ── USER PARAMETER ────────────────────────────────────────────────────────────
COLS = 6   # number of sample columns in the colour grid
# ─────────────────────────────────────────────────────────────────────────────

n    = RGB_mapped.shape[0]
rows = int(np.ceil(n / COLS))

print(f'Samples out of sRGB gamut: {(RGB.max(axis=1) > 1).sum()} of {n}')

# ── Part 1: Build colour-grid figure ─────────────────────────────────────────
cell_size = 1.0
gap       = 0.08
fig1, ax1 = plt.subplots(figsize=(COLS * 0.7, rows * 0.7))
ax1.set_xlim(0, COLS * cell_size)
ax1.set_ylim(0, rows * cell_size)
ax1.set_aspect('equal')
ax1.axis('off')

for idx in range(rows * COLS):
    row_i      = idx // COLS
    col_i      = idx % COLS
    col_mirror = COLS - 1 - col_i   # horizontal mirror to match physical layout

    x0 = col_mirror * cell_size + gap / 2
    y0 = (rows - 1 - row_i) * cell_size + gap / 2

    facecolor = RGB_mapped[idx] if idx < n else (1, 1, 1)

    rect = plt.Rectangle((x0, y0),
                          cell_size - gap, cell_size - gap,
                          facecolor=facecolor,
                          edgecolor='white', linewidth=4)
    ax1.add_patch(rect)

# Outer border
outer_rect = plt.Rectangle(
    (gap / 2, gap / 2),
    COLS * cell_size - gap, rows * cell_size - gap,
    facecolor='none', edgecolor='black', linewidth=4, zorder=10
)
ax1.add_patch(outer_rect)

# ── Capture figure as image BEFORE adding title (avoids text contaminating
#    the CIELAB channel maps computed in Part 2) ─────────────────────────────
buf = io.BytesIO()
fig1.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.05, dpi=120)
buf.seek(0)
imagen_capturada = plt.imread(buf)
imagen_sRGB = imagen_capturada[..., :3] if imagen_capturada.shape[-1] == 4 else imagen_capturada

# Now close the figure so it doesn't display as a separate image
plt.close(fig1)

# ── Part 2: sRGB → XYZ → CIELab for the captured image ──────────────────────
imagen_XYZ = colour.sRGB_to_XYZ(imagen_sRGB)
imagen_Lab = colour.XYZ_to_Lab(imagen_XYZ, illuminant_xy)

L = imagen_Lab[..., 0]
a = imagen_Lab[..., 1]
b = imagen_Lab[..., 2]

cmap_a = LinearSegmentedColormap.from_list('GreenRed',   ['green', 'white', 'red'])
cmap_b = LinearSegmentedColormap.from_list('BlueYellow', ['blue',  'white', 'yellow'])

fig2, axes = plt.subplots(2, 2, figsize=(12, 11))

axes[0, 0].imshow(imagen_sRGB)
axes[0, 0].set_title('Reconstructed sRGB Image')
axes[0, 0].axis('off')

img_L = axes[0, 1].imshow(L, cmap='gray', vmin=0, vmax=100)
axes[0, 1].set_title('L* channel (Luminance)')
axes[0, 1].axis('off')
fig2.colorbar(img_L, ax=axes[0, 1], fraction=0.046, pad=0.04)

img_a = axes[1, 0].imshow(a, cmap=cmap_a, vmin=-80, vmax=80)
axes[1, 0].set_title('a* channel (Green ↔ Red)')
axes[1, 0].axis('off')
fig2.colorbar(img_a, ax=axes[1, 0], fraction=0.046, pad=0.04)

img_b = axes[1, 1].imshow(b, cmap=cmap_b, vmin=-80, vmax=80)
axes[1, 1].set_title('b* channel (Blue ↔ Yellow)')
axes[1, 1].axis('off')
fig2.colorbar(img_b, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.suptitle(f'Decomposition of the reconstructed image into CIELAB channels', y=0.98)
plt.tight_layout()

out_dir  = 'results/graphics/LAB'
os.makedirs(out_dir, exist_ok=True)
svg_path = f'{out_dir}/cielab_{file_name}.svg'
plt.savefig(svg_path, format='svg', bbox_inches='tight')
print(f'Diagram saved: {svg_path}')
plt.show()

## Section 13 – sRGB Visualization (Flat Grid)

Renders each sample as a colour swatch arranged in a flat rectangular grid (no gaps between cells). Useful for a quick visual overview when there are many samples.

> **What to change:** `COLS` – number of sample columns in the grid.

In [ ]:
# ── USER PARAMETER ────────────────────────────────────────────────────────────
COLS = 6   # columns in the flat grid  (<-- adjust to your sample layout)
# ─────────────────────────────────────────────────────────────────────────────

n    = RGB_mapped.shape[0]
rows = int(np.ceil(n / COLS))

# Pad with white to fill the last row
pad = rows * COLS - n
img_data = np.vstack([RGB_mapped, np.ones((pad, 3))]) if pad > 0 else RGB_mapped.copy()

# Reshape to (rows, cols, 3) and apply horizontal mirror
img = np.fliplr(img_data.reshape(rows, COLS, 3))

print(f'Samples out of sRGB gamut: {(RGB.max(axis=1) > 1).sum()} of {n}')

fig, ax = plt.subplots(figsize=(COLS * 0.45, rows * 0.45))
ax.imshow(img, aspect='equal', interpolation='nearest')
ax.axis('off')
ax.set_title(f'Reconstructed sRGB Image')
plt.tight_layout()
plt.show()

## Section 14 – sRGB Visualization with Separated Cells and Border

Like Section 13 but with visible gaps between cells and a black outer border. This layout mimics a physical colour chart and is better suited for publication figures.

The figure is saved as an SVG to `results/graphics/sRGB/`.

> **What to change:**
> - `COLS` – number of sample columns.
> - `GAP` – fraction of cell size used as separator (default 0.08).
> - `CELL_PX` – cell size in inches per cell for figure sizing.

In [ ]:
# ── USER PARAMETERS ───────────────────────────────────────────────────────────
COLS    = 6     # number of sample columns in the grid
GAP     = 0.08  # gap between cells as a fraction of cell_size
CELL_PX = 0.7   # figure width/height per cell (inches)
# ─────────────────────────────────────────────────────────────────────────────

n         = RGB_mapped.shape[0]
rows      = int(np.ceil(n / COLS))
cell_size = 1.0

print(f'Samples out of sRGB gamut: {(RGB.max(axis=1) > 1).sum()} of {n}')

fig, ax = plt.subplots(figsize=(COLS * CELL_PX, rows * CELL_PX))
ax.set_xlim(0, COLS * cell_size)
ax.set_ylim(0, rows * cell_size)
ax.set_aspect('equal')
ax.axis('off')

for idx in range(rows * COLS):
    row_i      = idx // COLS
    col_i      = idx % COLS
    col_mirror = COLS - 1 - col_i

    x0 = col_mirror * cell_size + GAP / 2
    y0 = (rows - 1 - row_i) * cell_size + GAP / 2

    facecolor = RGB_mapped[idx] if idx < n else (1, 1, 1)

    rect = plt.Rectangle(
        (x0, y0), cell_size - GAP, cell_size - GAP,
        facecolor=facecolor, edgecolor='white', linewidth=4
    )
    ax.add_patch(rect)

# Black outer border
outer_rect = plt.Rectangle(
    (GAP / 2, GAP / 2),
    COLS * cell_size - GAP, rows * cell_size - GAP,
    facecolor='none', edgecolor='black', linewidth=4, zorder=10
)
ax.add_patch(outer_rect)
ax.set_title(f'Reconstructed sRGB Image')

out_dir  = 'results/graphics/sRGB'
os.makedirs(out_dir, exist_ok=True)
svg_path = f'{out_dir}/image_sRGB_{file_name}.svg'
plt.savefig(svg_path, format='svg', bbox_inches='tight')
print(f'Diagram saved: {svg_path}')
plt.tight_layout()
plt.show()

## Section 15 – sRGB Visualization with Matrix Positions

Same separated-cell layout as Section 14 but also prints the sample index inside each swatch. Useful when you need to cross-reference a visual with a specific sample number in the data tables.

> **What to change:** `COLS` and font size (`FONT_SIZE`) as needed.

In [ ]:
# ── USER PARAMETERS ───────────────────────────────────────────────────────────
COLS      = 6    # columns in the grid
FONT_SIZE = 5    # font size for sample index labels
# ─────────────────────────────────────────────────────────────────────────────

n         = RGB_mapped.shape[0]
rows      = int(np.ceil(n / COLS))
cell_size = 1.0
gap       = 0.08

print(f'Samples out of sRGB gamut: {(RGB.max(axis=1) > 1).sum()} of {n}')

fig, ax = plt.subplots(figsize=(COLS * 0.7, rows * 0.7))
ax.set_xlim(0, COLS * cell_size)
ax.set_ylim(0, rows * cell_size)
ax.set_aspect('equal')
ax.axis('off')

for idx in range(rows * COLS):
    row_i      = idx // COLS
    col_i      = idx % COLS
    col_mirror = COLS - 1 - col_i

    x0 = col_mirror * cell_size + gap / 2
    y0 = (rows - 1 - row_i) * cell_size + gap / 2
    w  = cell_size - gap
    h  = cell_size - gap

    if idx < n:
        facecolor = RGB_mapped[idx]
        # Choose white or black label for legibility
        brightness   = 0.299 * facecolor[0] + 0.587 * facecolor[1] + 0.114 * facecolor[2]
        label_colour = 'white' if brightness < 0.5 else 'black'
        ax.text(x0 + w / 2, y0 + h / 2, str(idx),
                ha='center', va='center',
                fontsize=FONT_SIZE, color=label_colour)
    else:
        facecolor = (1, 1, 1)

    rect = plt.Rectangle(
        (x0, y0), w, h,
        facecolor=facecolor, edgecolor='white', linewidth=4
    )
    ax.add_patch(rect)

# Black outer border
outer_rect = plt.Rectangle(
    (gap / 2, gap / 2),
    COLS * cell_size - gap, rows * cell_size - gap,
    facecolor='none', edgecolor='black', linewidth=4, zorder=10
)
ax.add_patch(outer_rect)
ax.set_title(f'Reconstructed sRGB Image with Sample Indices')

plt.tight_layout()
plt.show()

## Section 16 – Calculate Color Difference ($\Delta E^*_{00}$)

Calculates the perceptual color difference ($\Delta E^*_{00}$) between samples and a set of reference colors using the CIEDE2000 formula.

You can group samples (e.g., 6 samples per color) and compare them against a specific HEX code or CIELAB value.

> **What to change:**
> - `GROUP_SIZE` – Number of consecutive samples belonging to the same color.
> - `REFERENCE_COLORS` – Dictionary of reference values. Can be HEX strings (e.g. `"#ee3028"`) or CIELAB arrays (e.g. `[50.0, 0.0, 0.0]`).
> - `DISPLAY_LABELS` – Friendly names for the colors to be used in the exported tables.


In [ ]:
# ── USER PARAMETERS ───────────────────────────────────────────────────────────
GROUP_SIZE = 6

# Reference Colors (Can be HEX string or [L, a, b] list/array)
REFERENCE_COLORS = {
    "Color_1": "#ee3028",
    "Color_2": "#fef055",
    "Color_3": "#0f479a",
    "Color_4": "#63bc52",
    "Color_5": "#a666aa",
    "Color_6": "#8dd3d8",
}

DISPLAY_LABELS = {
    "Color_1": "Red",
    "Color_2": "Yellow",
    "Color_3": "Blue",
    "Color_4": "Green",
    "Color_5": "Magenta",
    "Color_6": "Cyan",
}
# ─────────────────────────────────────────────────────────────────────────────

def parse_reference(ref_val):
    if isinstance(ref_val, str) and ref_val.startswith('#'):
        # Convert HEX to RGB [0, 1]
        rgb = colour.notation.HEX_to_RGB(ref_val)
        # Convert sRGB to XYZ
        xyz = colour.sRGB_to_XYZ(rgb)
        # Convert XYZ to CIELAB using D65 illuminant (defined in Section 8)
        lab = colour.XYZ_to_Lab(xyz, illuminant=illuminant_xy)
        return lab
    elif isinstance(ref_val, (list, tuple, np.ndarray)) and len(ref_val) == 3:
        return np.array(ref_val)
    else:
        return np.array([np.nan, np.nan, np.nan])

detailed_data = []

# Sample_labels and Lab arrays are already in memory from previous sections
for index, label in enumerate(sample_labels):
    grupo_idx = (index // GROUP_SIZE) + 1
    grupo_name = f"Color_{grupo_idx}"
    
    if grupo_name not in REFERENCE_COLORS:
        continue
        
    ref_val = REFERENCE_COLORS[grupo_name]
    lab_ref = parse_reference(ref_val)
    lab_meas = Lab[index]
    
    # Calculate Delta E 2000
    try:
        de00 = colour.difference.delta_E_CIE2000(lab_meas, lab_ref)
    except Exception:
        de00 = np.nan
        
    detailed_data.append({
        'Color': DISPLAY_LABELS.get(grupo_name, grupo_name),
        'Position': label,
        'L*': lab_meas[0],
        'a*': lab_meas[1],
        'b*': lab_meas[2],
        'Reference_Input': str(ref_val),
        'L*_ref': lab_ref[0],
        'a*_ref': lab_ref[1],
        'b*_ref': lab_ref[2],
        'ΔE2000': de00
    })

df_detailed = pd.DataFrame(detailed_data)

# Create output directory and save detailed data
out_dir_de = 'results/spaces_values/delta_e00'
os.makedirs(out_dir_de, exist_ok=True)

detailed_output_path = f'{out_dir_de}/DeltaE_Details_{file_name}.xlsx'
df_detailed.to_excel(detailed_output_path, index=False)
print(f"Detailed Delta E saved: {detailed_output_path}")

# Generate and save summary statistics
summary_data = []
for grupo, group_df in df_detailed.groupby('Color', sort=False):
    ref_input = group_df['Reference_Input'].iloc[0]
    summary_data.append({
        'Color': grupo,
        'Reference_Input': ref_input,
        'Mean_ΔE00': group_df['ΔE2000'].mean(),
        'Std_ΔE00': group_df['ΔE2000'].std(),
        'Min_ΔE00': group_df['ΔE2000'].min(),
        'Max_ΔE00': group_df['ΔE2000'].max(),
    })

df_summary = pd.DataFrame(summary_data)
summary_output_path = f'{out_dir_de}/Summary_DeltaE_{file_name}.xlsx'
df_summary.to_excel(summary_output_path, index=False)
print(f"Summary Delta E saved: {summary_output_path}")

print("\nDelta E 00 Summary:")
display(df_summary)
